In [1]:
import os

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import ttest_ind, f_oneway, kruskal, pearsonr, spearmanr, shapiro, levene


import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the datasets
train_df = pd.read_csv('../data/experiments/train/train_data.csv')

In [4]:
categorical_cols = ['Store_Type','Location_Type','Region_Code','Holiday','Discount']
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')

train_df['Date'] = pd.to_datetime(train_df['Date'])

In [5]:
train_df.head()

,ID,Store_id,Store_Type,Location_Type,Region_Code,Date,Holiday,Discount,#Order,Sales
0,T1000001,1,S1,L3,R1,2018-01-01,1,Yes,9,7011.84
1,T1000115,2,S3,L1,R3,2018-01-01,1,Yes,25,18137.52
2,T1000128,3,S4,L2,R1,2018-01-01,1,Yes,72,57288.00
3,T1000097,4,S1,L1,R2,2018-01-01,1,Yes,80,53615.52
4,T1000133,5,S1,L1,R3,2018-01-01,1,Yes,47,36316.08


In [6]:
train_df['Discount'] = train_df['Discount'].astype(str)
train_df['Holiday'] = train_df['Holiday'].astype(int)

In [7]:
def interpret(p_value, alpha=0.05):
    return "Reject H0 ✅ (Significant)" if p_value < alpha else "Fail to Reject H0 ❌ (Not Significant)"

In [8]:
ALPHA = 0.05
print("\n=== 1. Discount Impact on Sales ===")
print("H0: Mean Sales (Discount = Yes) = Mean Sales (Discount = No)")
print("H1: Mean Sales (Discount = Yes) > Mean Sales (Discount = No)")
print("Alpha:", ALPHA)

discount_yes = train_df[train_df['Discount'] == 'Yes']['Sales']
discount_no = train_df[train_df['Discount'] == 'No']['Sales']
_, p_var = levene(discount_yes, discount_no)
print("Levene's Test P-value:", p_var)
# Welch’s t-test
t_stat, p_val = ttest_ind(discount_yes, discount_no, equal_var=False, alternative='greater')

print("T-stat:", t_stat)
print("P-value:", p_val)
print("Conclusion:", interpret(p_val))


=== 1. Discount Impact on Sales ===
H0: Mean Sales (Discount = Yes) = Mean Sales (Discount = No)
H1: Mean Sales (Discount = Yes) > Mean Sales (Discount = No)
Alpha: 0.05
Levene's Test P-value: 2.7856500484311e-310
T-stat: 129.804762071436
P-value: 0.0
Conclusion: Reject H0 ✅ (Significant)


In [9]:
ALPHA = 0.05
print("\n=== 2. Holiday Effect on Sales ===")
print("H0: Mean Sales (Holiday) = Mean Sales (Non-Holiday)")
print("H1: Mean Sales (Holiday) > Mean Sales (Non-Holiday)")
print("Alpha:", ALPHA)

holiday = train_df[train_df['Holiday'] == 1]['Sales']
non_holiday = train_df[train_df['Holiday'] == 0]['Sales']
_, p_var = levene(holiday, non_holiday)
print("Levene's Test P-value:", p_var)
# One-sided test 
t_stat, p_val = ttest_ind(holiday, non_holiday, equal_var=False, alternative='greater')

print("T-stat:", t_stat)
print("P-value (one-sided):", p_val)
print("Conclusion:", interpret(p_val))


=== 2. Holiday Effect on Sales ===
H0: Mean Sales (Holiday) = Mean Sales (Non-Holiday)
H1: Mean Sales (Holiday) > Mean Sales (Non-Holiday)
Alpha: 0.05
Levene's Test P-value: 1.0713509165537466e-20
T-stat: -64.06898038306029
P-value (one-sided): 1.0
Conclusion: Fail to Reject H0 ❌ (Not Significant)


In [10]:
ALPHA = 0.05
print("\n=== 3. Store Type vs Sales (ANOVA) ===")
print("H0: All Store Types have equal mean sales")
print("H1: At least one Store Type has different mean sales")
print("Alpha:", ALPHA)

groups = [group['Sales'].values for name, group in train_df.groupby('Store_Type')]

f_stat, p_val = f_oneway(*groups)

print("F-stat:", f_stat)
print("P-value:", p_val)
print("Conclusion:", interpret(p_val))


=== 3. Store Type vs Sales (ANOVA) ===
H0: All Store Types have equal mean sales
H1: At least one Store Type has different mean sales
Alpha: 0.05
F-stat: 26936.769791639887
P-value: 0.0
Conclusion: Reject H0 ✅ (Significant)


In [11]:
ALPHA = 0.05

print("\n=== 4. Regional Sales Variability ===")
print("H0: Sales distribution is same across regions")
print("H1: At least one region has different sales distribution")
print("Alpha:", ALPHA)

normal = True
normality_results = {}
region_groups = []

for name, group in train_df.groupby('Region_Code'):
    
    sales = group['Sales'].dropna()
    n = len(sales)
    
    # Store group for final test
    region_groups.append(sales.values)
    
    # Handle small samples
    if n < 3:
        normality_results[name] = {
            "sample_size": n,
            "p_value": None,
            "normal": False,
            "remark": "Too few samples → Treated as NON-NORMAL"
        }
        normal = False
        continue
    
    # Sampling for large data
    sample = sales.sample(min(5000, n), random_state=42)
    
    # Shapiro test
    stat, p = shapiro(sample)
    
    is_normal = p > ALPHA
    
    normality_results[name] = {
        "sample_size": n,
        "p_value": p,
        "normal": is_normal
    }
    
    if not is_normal:
        normal = False

# Print normality results
print("\n--- Normality Check ---")
for region, res in normality_results.items():
    print(f"{region}: n={res['sample_size']}, p={res['p_value']}, normal={res['normal']}")

# Select test
if normal:
    stat, p_val = f_oneway(*region_groups)
    test_used = "ANOVA"
else:
    stat, p_val = kruskal(*region_groups)
    test_used = "Kruskal-Wallis"

# Final output
print("\n--- Test Result ---")
print("Test Used:", test_used)
print("Statistic:", stat)
print("P-value:", p_val)
print("Conclusion:", interpret(p_val, ALPHA))


=== 4. Regional Sales Variability ===
H0: Sales distribution is same across regions
H1: At least one region has different sales distribution
Alpha: 0.05

--- Normality Check ---
R1: n=49104, p=2.1298311390361207e-41, normal=False
R2: n=41580, p=8.664342748590458e-35, normal=False
R3: n=34056, p=1.0446634892443505e-32, normal=False
R4: n=19800, p=1.10306481411826e-40, normal=False

--- Test Result ---
Test Used: Kruskal-Wallis
Statistic: 2752.7327638661177
P-value: 0.0
Conclusion: Reject H0 ✅ (Significant)


In [12]:
ALPHA = 0.05

print("\n=== 5. Orders vs Sales Correlation ===")
print("H0: No correlation between Orders and Sales (ρ = 0)")
print("H1: Significant correlation exists (ρ ≠ 0)")
print("Alpha:", ALPHA)

orders = train_df['#Order']
sales = train_df['Sales']

# Normality check
_, p_orders = shapiro(orders.sample(min(5000, len(orders))))
_, p_sales = shapiro(sales.sample(min(5000, len(sales))))

if p_orders > 0.05 and p_sales > 0.05:
    corr, p_val = pearsonr(orders, sales)
    method = "Pearson"
else:
    corr, p_val = spearmanr(orders, sales)
    method = "Spearman"

print("Method:", method)
print("Correlation:", corr)
print("P-value:", p_val)
print("Conclusion:", interpret(p_val))


=== 5. Orders vs Sales Correlation ===
H0: No correlation between Orders and Sales (ρ = 0)
H1: Significant correlation exists (ρ ≠ 0)
Alpha: 0.05
Method: Spearman
Correlation: 0.9353952802079869
P-value: 0.0
Conclusion: Reject H0 ✅ (Significant)
